# MorphoNet-BT — 03: MRI Preprocessing and Morphology
## Objective and method
Inspect intensity, compare per-image normalization and prepare grayscale processing aligned with region/boundary/distance masks. Approved splits are read and hash-checked, not changed; no models or training. Full-cohort statistics are descriptive. The initial choice uses fold-1 Train and transform mathematics, not validation/test accuracy or cohort-wide normalization estimates.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.


## Configuration, imports and environment

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
from pathlib import Path
from copy import deepcopy
import sys, json, platform
PROJECT_ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
assert Path(sys.prefix).name.lower() == "dl", sys.executable
import numpy as np
import pandas as pd
import torch
import cv2, scipy, yaml
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from torch.utils.data import DataLoader
from src.data.kfold import load_manifest, reload_splits, audit_splits
from src.data.dataset import BrainTumorDataset, verify_frozen_splits, load_checked_sample, ORIGINAL_TO_INDEX, INDEX_TO_CLASS
from src.data.preprocessing import DEFAULT_CONFIG, normalize_image, resize_pair, preprocess
from src.data.morphology import morphology_targets, validate_targets
from src.data.preprocessing_audit import scan_intensities, audit_pipeline, numerical_checks, border_sanity
from src.data.preprocessing_plots import (choose_examples, normalization_figure, distribution_figure, morphology_figure, augmentation_figure)
SEED = 42
OUTPUT = PROJECT_ROOT / "results/preprocessing"
MORPHOLOGY_OUTPUT = PROJECT_ROOT / "results/morphology_examples"
AUGMENTATION_OUTPUT = OUTPUT / "augmentation_examples"
for folder in (OUTPUT, MORPHOLOGY_OUTPUT, AUGMENTATION_OUTPUT): folder.mkdir(parents=True, exist_ok=True)
print("Python:", sys.executable)
print("NumPy:", np.__version__, "OpenCV:", cv2.__version__, "SciPy:", scipy.__version__)

## Frozen splits and dataset summary
Verify SHA-256 for all fifteen CSVs, PID intersections and sample coverage without generating new splits.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
frozen_before = verify_frozen_splits(PROJECT_ROOT)
frame = load_manifest(PROJECT_ROOT / "results/dataset_statistics.csv")
saved_splits = reload_splits(PROJECT_ROOT / "splits")
split_audit, coverage = audit_splits(saved_splits, frame)
display(split_audit)
display(pd.Series(coverage))
train_frame = saved_splits[1]["train"]
train_ids = set(train_frame.sample_id)
examples = choose_examples(train_frame)
display(frame.groupby(["class_name", "image_shape"]).size().rename("slices"))

## Full intensity audit
Inspect raw minimum/maximum/mean/std, P1/P50/P95/P99/P99.5/P99.9 and zero background. Measure max/P99, max/P99.9, fractions above P99.9 and 10,000; the latter is descriptive, not a clipping threshold. Excess energy above P99.9 relative to variance energy describes the upper tail, not measurement error. Calculate Min-Max/Z-score statistics after 256×256 resizing.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
intensity = scan_intensities(frame, OUTPUT)
train_intensity = intensity[intensity.sample_id.isin(train_ids)]
columns = ["minimum", "maximum", "mean", "std", "p99", "p999", "max_over_p99", "max_over_p999", "zero_fraction", "p999_excess_variance_fraction"]
display(intensity[columns].describe())
display(train_intensity[columns].describe())
intensity.groupby("class_name")[columns].agg(["mean", "median", "min", "max"]).to_csv(OUTPUT / "intensity_by_class.csv")
patient_intensity = intensity.groupby(["PID", "class_name"])[columns].mean()
patient_intensity.to_csv(OUTPUT / "intensity_by_patient.csv")
display(patient_intensity.describe())
tail_report = {"images_above_10000": int((intensity.maximum > 10000).sum()),
               "median_max_over_p999": float(intensity.max_over_p999.median()),
               "max_max_over_p999": float(intensity.max_over_p999.max()),
               "train_std_min": float(train_intensity["std"].min()),
               "train_std_max": float(train_intensity["std"].max()),
               "train_median_max_over_p99": float(train_intensity.max_over_p99.median()),
               "max_p999_excess_variance_fraction": float(intensity.p999_excess_variance_fraction.max())}
display(pd.Series(tail_report))
(OUTPUT / "intensity_tail_report.json").write_text(json.dumps(tail_report, indent=2), encoding="utf-8")

## Class/patient distributions and histograms

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
distribution_figure(intensity, OUTPUT)
plt.show()
normalization_figure(examples, OUTPUT)
plt.show()

## Normalization comparison and rationale
Min-Max: `(x-min)/(max-min)` gives [0,1], but the maximum pixel controls scale and image standard deviations vary. Z-score: `(x-mean)/std` over all final-image pixels, including background, ddof=0, gives approximately zero mean/unit SD before intensity augmentation. Both are linear, preserve intensity ordering/relative contrast and use no mask. Each displayed panel spans its own full range; visual similarity is expected and does not imply recovered information.
The initial recommendation is per-image Z-score without clipping. Variable Train intensity/contrast motivates scale standardization without relying solely on the maximum or truncating potentially high enhancement; it does not establish higher accuracy. Outliers and background fraction still affect Z-score, and statistics do not align tissue across scanners. No validated clipping threshold was available. No learned parameters or cohort/Validation/Test normalization statistics are used.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
comparison = train_intensity[[c for c in intensity if c.startswith("minmax_") or c.startswith("zscore_")]].describe()
display(comparison)
comparison.to_csv(OUTPUT / "normalization_comparison_statistics.csv")
config = deepcopy(DEFAULT_CONFIG)
config["normalization"]["method"] = "zscore"
assert config["seed"] == SEED
display(config)

## Processing order and resize methods
1. Load MRI/mask; check PID, label and image hash against approved records. Convert {0,255} to {0,1}; other mask values are errors.
2. Train only: apply one shared geometric matrix at original resolution.
3. Resize to 256×256: MRI cv2.INTER_LINEAR; mask cv2.INTER_NEAREST_EXACT, with compatible pixel-center convention. warpAffine uses INTER_NEAREST for masks.
4. Skip interpolation if already 256×256. Normalize MRI; apply Train-only gain/noise.
5. Verify final binary mask and generate all morphology targets from it, rather than transforming old target maps.
Early binary validation changes representation, not geometry. Processing runs on CPU; the full dataset is not moved to GPU.

## Region, boundary and distance targets
Region is the final binary mask. Boundary is `M AND NOT erosion(M)`: one erosion with a 3×3 square and background outside the image. This is a one-layer inner boundary under 8-neighborhood connectivity; diagonal Euclidean width differs from axial width. No dilation or thick boundary is imposed. Distance is the internal Euclidean distance transform with one-pixel zero padding, divided by each mask's maximum; outside is zero. Inner-boundary pixels may be small positive values; a single-pixel tumor has value 1. All maps are float32 `[1,256,256]`. Distances refer to resized-image pixels, not millimeters or 3D morphology.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
morphology_figure(examples, config, MORPHOLOGY_OUTPUT)
plt.show()

## Frozen training augmentation
Geometry probability 0.8: rotation ±10°, translation up to ±2% per axis, scale 0.95–1.05. No elastic deformation, intentional crops or flips; horizontal flipping lacked anatomical justification. Reject proposals moving any mask-convex-hull point outside the image; retry up to 20 times, then explicitly record identity fallback. This neither excludes patients nor repairs invalid masks; invalid samples raise errors. MRI-only post-normalization gain 0.95–1.05 with probability 0.5; Gaussian noise SD 0.01 in normalized units with probability 0.3. No clipping or subsequent renormalization that would cancel augmentation. Validation/Test remain deterministic even with augmentation.enabled=True.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
augmentation_figure(examples, config, AUGMENTATION_OUTPUT)
plt.show()

## Alignment and edge-case checks
Synthetic known shapes test translation, rotation/scale and resize, plus binary/empty/single-pixel/edge-touching cases. tumorBorder checks cover only six examples; rasterization differences are not automatically annotation errors.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
checks = numerical_checks(config)
display(pd.Series(checks))
border_checks = examples[["sample_id", "PID", "class_name"]].copy()
border_checks["native_border_polygon_mask_dice"] = [border_sanity(row) for row in examples.to_dict("records")]
border_checks.to_csv(OUTPUT / "border_alignment_examples.csv", index=False)
display(border_checks)

## BrainTumorDataset and DataLoader
mode is mandatory and matches the CSV name. Outputs: image/label/pid/sample_id and region/boundary/distance when include_targets=True. Mapping: 1→0 Meningioma, 2→1 Glioma, 3→2 Pituitary; classifier/probability columns must agree. Random draws depend on seed/sample_id/epoch, not worker arrival order. Call dataset.set_epoch(epoch) before each iterator and retain persistent_workers=False so workers receive the new epoch. Repeated access within an epoch deliberately returns the same transform.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
datasets = {mode: BrainTumorDataset(PROJECT_ROOT / f"splits/fold_1_{mode}.csv", mode=mode, config=config, seed=SEED)
            for mode in ("train", "val", "test")}
batch_shapes = {}
for mode, dataset in datasets.items():
    loader = DataLoader(dataset, batch_size=4, shuffle=mode == "train", num_workers=0,
                        generator=torch.Generator().manual_seed(SEED))
    batch = next(iter(loader))
    for key in ("image", "region", "boundary", "distance"):
        assert batch[key].shape == (4, 1, 256, 256)
        assert batch[key].dtype == torch.float32 and torch.isfinite(batch[key]).all()
    assert batch["label"].shape == (4,) and batch["label"].dtype == torch.long
    assert set(batch["label"].tolist()) <= {0, 1, 2}
    batch_shapes[mode] = {k: list(v.shape) if torch.is_tensor(v) else len(v) for k, v in batch.items()}
    print(mode, batch_shapes[mode])
display({"original_to_index": ORIGINAL_TO_INDEX, "index_to_class": INDEX_TO_CLASS})

## Deterministic validation/test, repeatable training and mask-free MRI processing

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
for mode in ("val", "test"):
    a, b = datasets[mode][0], datasets[mode][0]
    assert all(torch.equal(a[k], b[k]) for k in ("image", "label", "region", "boundary", "distance"))
train_dataset = datasets["train"]
first = train_dataset[0]
assert torch.equal(first["image"], train_dataset[0]["image"])
train_dataset.set_epoch(1)
changed = train_dataset[0]
assert not torch.equal(first["image"], changed["image"])
train_dataset.set_epoch(0)
assert torch.equal(first["image"], train_dataset[0]["image"])
record = saved_splits[1]["test"].iloc[0].to_dict()
raw_image, _ = load_checked_sample(record, include_mask=False)
image_only, _ = preprocess(raw_image, mode="test", config=config)
assert set(image_only) == {"image"}
assert np.array_equal(image_only["image"], datasets["test"][0]["image"].numpy())
checks.update(dataset_val_test_deterministic=True, train_epoch_reproducibility=True, image_only_matches_supervised=True)
print("PASS: deterministic evaluation, epoch reproducibility, MRI-only preprocessing")

## Full sample checks
Check all 3,064 image/mask pairs deterministically, including fifteen native 256×256 images. Check one fixed random augmentation draw for each fold-1 Train image (1,942), never Validation/Test augmentation. Errors stop execution and record sample ID/PID/class/file/type. Passing these draws does not guarantee every future draw; runtime checks remain active.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
integrity = audit_pipeline(frame, config, OUTPUT, train_ids)
assert len(integrity) == 3064 and integrity.status.eq("PASS").all()
assert int(integrity.native_height.eq(256).sum()) == 15
assert int(integrity.augmentation_tested.sum()) == 1942
display(integrity[["region_pixels", "boundary_pixels", "boundary_fraction", "distance_min", "distance_max", "geometry_attempts"]].describe())
audit_summary = {"deterministic_samples_passed": len(integrity), "augmented_train_samples_passed": int(integrity.augmentation_tested.sum()),
                "native_256_samples_passed": int(integrity.native_height.eq(256).sum()),
                "failures": 0, "identity_fallbacks": int(integrity.geometry_identity_fallback.sum()),
                "resampled_geometry_cases": int(integrity.geometry_attempts.gt(1).sum()),
                "min_region_pixels": int(integrity.region_pixels.min()), "min_boundary_pixels": int(integrity.boundary_pixels.min()),
                "max_boundary_region_fraction": float(integrity.boundary_fraction.max()),
                "frozen_splits_unchanged": verify_frozen_splits(PROJECT_ROOT) == frozen_before,
                "batch_shapes": batch_shapes, "checks": checks}
assert audit_summary["frozen_splits_unchanged"]
display(pd.Series({k: v for k, v in audit_summary.items() if k not in ("batch_shapes", "checks")}))

## Save preprocessing configuration and final report
Save preprocessing settings without training parameters or model results. Phase 3 does not write split CSVs.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
(PROJECT_ROOT / "configs").mkdir(exist_ok=True)
(PROJECT_ROOT / "configs/default.yaml").write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
loaded_config = yaml.safe_load((PROJECT_ROOT / "configs/default.yaml").read_text(encoding="utf-8"))
assert loaded_config == config
(OUTPUT / "validation_report.json").write_text(json.dumps(audit_summary, indent=2), encoding="utf-8")
(OUTPUT / "frozen_split_checksums.json").write_text(json.dumps(frozen_before, indent=2), encoding="utf-8")
(OUTPUT / "versions.json").write_text(json.dumps({"python": platform.python_version(), "numpy": np.__version__, "torch": torch.__version__, "opencv": cv2.__version__, "scipy": scipy.__version__, "seed": SEED}, indent=2), encoding="utf-8")
report = f"""Phase 3: MRI preprocessing and morphology targets — no training.
Recommendation: per-final-image Z-score over all pixels, ddof=0, no clipping/global statistics.
Min-Max and Z-score were compared statistically/visually; variable Train scales and avoiding maximum-only scaling motivate the initial choice, not an accuracy claim.
Background/outliers affect Z-score; no justified robust-clipping threshold was tested.
Resize: bilinear MRI, nearest-neighbor exact masks, no interpolation at 256×256.
Shared affine at original resolution: ±10° rotation, ±2% translation, scale0.95–1.05, probability0.8.
Reject tumor clipping up to20 attempts, then record identity. Horizontal flip disabled without anatomical justification.
Post-normalization MRI gain0.95–1.05 probability0.5; Gaussian std0.01 probability0.3.
Binary masks: convert0/255, reject other values/empty masks.
Boundary: M minus one 3×3 square erosion, one inner 8-neighbor layer.
Distance: internal pixel-unit EDT, zero padding, per-mask max division, zero outside tumor.
Labels: 1=Meningioma→0; 2=Glioma→1; 3=Pituitary→2.
DataLoader batch4; image/maps [4,1,256,256] float32; labels [4] int64; pid/sample_id strings.
Deterministic Val/Test; Train uses seed/sample_id/epoch; call set_epoch and keep persistent_workers=False.
Deterministic checks passed: {len(integrity)}/3064; Train augmentation: {audit_summary['augmented_train_samples_passed']}/1942; errors0.
Geometry resampling: {audit_summary['resampled_geometry_cases']}; identity fallbacks: {audit_summary['identity_fallbacks']}.
Smallest region: {audit_summary['min_region_pixels']} pixels; boundary: {audit_summary['min_boundary_pixels']} pixels.
All frozen split hashes match. Phase4 not executed.
Numerical checks establish transform/target implementation, not every annotation's anatomical correctness; examples also receive visual review.
"""
(OUTPUT / "preprocessing_summary.txt").write_text(report, encoding="utf-8")
(OUTPUT / "progress.json").write_text(json.dumps({"stage": "complete", "completed": 3064, "total": 3064}), encoding="utf-8")
print(report)

## Historical conclusion
Grayscale processing, morphology targets and Dataset/DataLoader were prepared with fail-fast checks and explicit reproducible settings. Data are read on CPU on demand; no full-dataset GPU loading or training. Preprocessing tests cannot establish anatomical annotation quality or cross-hospital generalization. The historical workflow stopped for researcher approval before Phase 4.

## Recorded findings and review notes
All twelve inspection cells passed: 3,064 processed images, including fifteen native 256×256 images. One fixed augmentation draw for each of 1,942 fold-1 Train images passed without resampling/identity fallback in that audit. Test batch `[4,1,256,256]`; images/maps float32, labels int64. num_workers=0 and 2 agreed in the separately recorded Windows worker_check.json. Smallest final mask: 64 pixels. Maximum boundary/region ratio: 67.9% for sample 1271 with 81 pixels; a one-layer boundary can occupy a large fraction of small/thin masks without extra erosion thickness. Incorrect PID and Test-as-Train access were rejected; all fifteen split hashes matched. Initial Z-score recommendation did not assert accuracy superiority. Train SD after Min-Max ranged 0.0835–0.2497; after Z-score it was approximately 1, with mean approximately zero before intensity augmentation. Distance units are pixels, not mm; these tests do not establish anatomy or enhancement. Historical enlarged examples were audit displays, not model-input ROIs. Dataset-derived images are omitted from this public copy pending reuse approval. Phase 4 had not begun at that historical handoff.